In [1]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from ann_model_massspecgym import AnnRetrieval
from distance_functions import cosine_similarity, euclidean_distance
from chemembed_transforms import ChemEmbedSpecTransform, molformer_factory
from massspecgym.data.datasets import MassSpecDataset
from massspecgym.data.data_module import MassSpecDataModule
from massspecgym.data.transforms import InMemCachedMolTransform

# 1) MoLFormer Embedding

In [2]:
# MoLFormer (768 dims)
mol_transform_factory = molformer_factory
embedding_name = "molformer"
mol_embedding_dim = 768

selected_molecular_embedding = InMemCachedMolTransform(
    mol_transform_factory=mol_transform_factory,
    verbose=True
)

Loading weights:   0%|          | 0/207 [00:00<?, ?it/s]

[transformers] MolformerModel LOAD REPORT from: ibm/MoLFormer-XL-both-10pct
Key                                | Status     |  | 
-----------------------------------+------------+--+-
lm_head.transform.LayerNorm.weight | UNEXPECTED |  | 
lm_head.transform.dense.weight     | UNEXPECTED |  | 
lm_head.transform.dense.bias       | UNEXPECTED |  | 
lm_head.decoder.weight             | UNEXPECTED |  | 
lm_head.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initializing InMemCachedMolTransform for MoLFormerMolTransform. Cache file not provided or missing; molecules will be transformed on the fly. Call `build_cache(smiles_list)` to create it.


## Calculate embeddings

In [4]:
dataset_path = Path("data/newMSG.tsv")
transformer = mol_transform_factory()

df = pd.read_csv(dataset_path, sep="\t")
df["molformer_embedding"] = df["smiles"].apply(transformer.from_smiles)
df

Loading weights:   0%|          | 0/207 [00:00<?, ?it/s]

[transformers] MolformerModel LOAD REPORT from: ibm/MoLFormer-XL-both-10pct
Key                                | Status     |  | 
-----------------------------------+------------+--+-
lm_head.transform.LayerNorm.weight | UNEXPECTED |  | 
lm_head.transform.dense.weight     | UNEXPECTED |  | 
lm_head.transform.dense.bias       | UNEXPECTED |  | 
lm_head.decoder.weight             | UNEXPECTED |  | 
lm_head.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,identifier,mzs,intensities,smiles,inchikey,formula,precursor_formula,parent_mass,precursor_mz,adduct,instrument_type,collision_energy,fold,simulation_challenge,molformer_embedding
0,MassSpecGymID0000201,"83.0491,123.0417,134.0964,141.0509,156.0781,20...","0.16216216216216217,0.04704704704704705,1.0,0....",CC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@...,GYSCAQFHASJXRS,C45H57N3O9,C45H57N3NaO9,783.408982,806.3982,[M+Na]+,Orbitrap,50.0,test,False,"[-0.089028426, 0.35098684, 0.5181631, 0.627438..."
1,MassSpecGymID0000202,"384.1765,545.2595,645.3115,806.3987","0.17917917917917917,0.05005005005005005,0.1941...",CC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@...,GYSCAQFHASJXRS,C45H57N3O9,C45H57N3NaO9,783.408982,806.3982,[M+Na]+,Orbitrap,30.0,test,False,"[-0.089028426, 0.35098684, 0.5181631, 0.627438..."


# 2) Load Dataset

In [5]:
test_dataset = MassSpecDataset(
    pth=dataset_path,
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=selected_molecular_embedding,
)

data_module_test = MassSpecDataModule(
    dataset=test_dataset,
    batch_size=1,
    num_workers=0
)

data_module_test.setup("test")

# 3) Model Evaluation

## Load model

In [6]:
model_name = "ANN_trained_molformer.ckpt"
model = AnnRetrieval.load_from_checkpoint(
    f"models/{model_name}",
    mol_embedding_dim=mol_embedding_dim
)

with torch.no_grad():
    print(model.eval())

c:\Users\danay\Documents\Code\MassSpecGym\.milab\Lib\site-packages\lightning_fabric\utilities\cloud_io.py:57: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


AnnRetrieval(
  (ann): ANN_Class(
    (fc1): Linear(in_features=70000, out_features=2048, bias=True)
    (fc2): Linear(in_features=2048, out_features=1024, bias=True)
    (fc3): Linear(in_features=1024, out_features=768, bias=True)
    (dropout): Dropout(p=0.25, inplace=False)
  )
)


## Predict MoLFormer Embeddings

In [7]:
model_predictions = []
with torch.no_grad():
    for entry in data_module_test.test_dataloader():
        input = entry["spec"]
        if (torch.isnan(input).any()):
            input = torch.from_numpy(np.array([0] * mol_embedding_dim).astype(np.float32))
        input = input.to("cpu") 
        outputs = model(input).cpu()
        model_predictions.append(outputs)

pred_df = pd.concat([df, pd.DataFrame({"molformer_embedding_pred" : model_predictions}).reset_index(drop=True)], axis=1)
pred_df

,identifier,mzs,intensities,smiles,inchikey,formula,precursor_formula,parent_mass,precursor_mz,adduct,instrument_type,collision_energy,fold,simulation_challenge,molformer_embedding,molformer_embedding_pred
0,MassSpecGymID0000201,"83.0491,123.0417,134.0964,141.0509,156.0781,20...","0.16216216216216217,0.04704704704704705,1.0,0....",CC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@...,GYSCAQFHASJXRS,C45H57N3O9,C45H57N3NaO9,783.408982,806.3982,[M+Na]+,Orbitrap,50.0,test,False,"[-0.089028426, 0.35098684, 0.5181631, 0.627438...","[[tensor(0.2645), tensor(0.2376), tensor(0.545..."
1,MassSpecGymID0000202,"384.1765,545.2595,645.3115,806.3987","0.17917917917917917,0.05005005005005005,0.1941...",CC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@...,GYSCAQFHASJXRS,C45H57N3O9,C45H57N3NaO9,783.408982,806.3982,[M+Na]+,Orbitrap,30.0,test,False,"[-0.089028426, 0.35098684, 0.5181631, 0.627438...","[[tensor(0.3937), tensor(0.3983), tensor(0.765..."


## Compare with real embedding

In [8]:
results = list()

cosine = cosine_similarity(pred_df, "molformer_embedding", "molformer_embedding_pred")
results.append(pd.DataFrame({'measure': "cosine", 'value': cosine}))
euc = euclidean_distance(pred_df, "molformer_embedding", "molformer_embedding_pred")
results.append(pd.DataFrame({'measure': "euclidean", 'value': euc}))

pd.concat(results)

,measure,value
0,cosine,0.764643
1,cosine,0.659604
0,euclidean,10.002392
1,euclidean,12.338811
